# Final evaluation and explanation

This notebook loads the validation-selected configuration, scores the locked gold test split once for the gold-only and expanded conditions, and generates the final analytical artifacts.


In [ ]:
import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lime.lime_text import LimeTextExplainer
from pyspark.ml.classification import (
    GBTClassificationModel, LinearSVCModel, LogisticRegressionModel,
    NaiveBayesModel, RandomForestClassificationModel
)
from pyspark.ml.feature import (
    CountVectorizerModel, IDFModel, VectorAssembler, Word2VecModel
)
from pyspark.ml.linalg import VectorUDT, Vectors
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import ArrayType, IntegerType, StringType, StructField, StructType
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score
)

BASE_DIR = Path("/kaggle/working/bangla_fake_review")
PROCESSED_DIR = BASE_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"
REPORT_DIR = BASE_DIR / "report"
AUDIT_DIR = BASE_DIR / "audit"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("BanglaFakeReview_FinalEvaluation")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

with open(MODELS_DIR / "final_selection.json", encoding="utf-8") as handle:
    final_manifest = json.load(handle)
selected_name = final_manifest["selected_model"]

model_loaders = {
    "logistic_regression": LogisticRegressionModel,
    "naive_bayes": NaiveBayesModel,
    "linear_svc": LinearSVCModel,
    "random_forest": RandomForestClassificationModel,
    "gbt": GBTClassificationModel,
}
model_class = model_loaders[selected_name]
gold_model = model_class.load(str(MODELS_DIR / f"{selected_name}_gold"))
expanded_model = model_class.load(str(MODELS_DIR / f"{selected_name}_expanded"))

test = spark.read.parquet(str(PROCESSED_DIR / "test_model_ready.parquet"))


In [ ]:
def evaluate_predictions(predictions, condition):
    columns = ["label", "prediction"]
    if "probability" in predictions.columns:
        score_column = "probability"
    else:
        score_column = "rawPrediction"
    columns.append(score_column)

    rows = predictions.select(*columns).collect()
    labels = np.asarray([int(row["label"]) for row in rows])
    predicted = np.asarray([int(row["prediction"]) for row in rows])
    fake_target = (labels == 0).astype(int)
    fake_score = np.asarray([float(row[score_column][0]) for row in rows])

    return {
        "condition": condition,
        "model": selected_name,
        "macro_f1": float(f1_score(labels, predicted, average="macro", zero_division=0)),
        "macro_precision": float(
            precision_score(labels, predicted, average="macro", zero_division=0)
        ),
        "macro_recall": float(
            recall_score(labels, predicted, average="macro", zero_division=0)
        ),
        "accuracy": float(accuracy_score(labels, predicted)),
        "fake_pr_auc": float(average_precision_score(fake_target, fake_score)),
        "fake_roc_auc": float(roc_auc_score(fake_target, fake_score)),
        "fake_precision": float(
            precision_score(labels, predicted, labels=[0], average="macro", zero_division=0)
        ),
        "fake_recall": float(
            recall_score(labels, predicted, labels=[0], average="macro", zero_division=0)
        ),
        "fake_f1": float(
            f1_score(labels, predicted, labels=[0], average="macro", zero_division=0)
        ),
        "genuine_precision": float(
            precision_score(labels, predicted, labels=[1], average="macro", zero_division=0)
        ),
        "genuine_recall": float(
            recall_score(labels, predicted, labels=[1], average="macro", zero_division=0)
        ),
        "genuine_f1": float(
            f1_score(labels, predicted, labels=[1], average="macro", zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(labels, predicted, labels=[0, 1]).tolist(),
    }

gold_predictions = gold_model.transform(test)
expanded_predictions = expanded_model.transform(test)
test_metrics = [
    evaluate_predictions(gold_predictions, "gold_only"),
    evaluate_predictions(expanded_predictions, "gold_plus_pseudo"),
]
test_metrics_frame = pd.DataFrame(test_metrics)
test_metrics_frame.to_csv(RESULTS_DIR / "test_results_final.csv", index=False)

gold_predictions.select(
    "review_id", "label", "prediction", "rawPrediction"
).write.mode("overwrite").parquet(
    str(RESULTS_DIR / "test_predictions_gold.parquet")
)
expanded_predictions.select(
    "review_id", "label", "prediction", "rawPrediction"
).write.mode("overwrite").parquet(
    str(RESULTS_DIR / "test_predictions_expanded.parquet")
)

print(test_metrics_frame[
    ["condition", "model", "macro_f1", "fake_pr_auc", "fake_f1", "accuracy"]
].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, result in zip(axes, test_metrics):
    sns.heatmap(
        np.asarray(result["confusion_matrix"]),
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=axis,
    )
    axis.set(
        xlabel="Predicted label",
        ylabel="True label",
        title=result["condition"].replace("_", " ").title(),
    )
fig.tight_layout()
fig.savefig(REPORT_DIR / "confusion_matrices.png", dpi=200)
plt.close(fig)

metric_plot = test_metrics_frame.melt(
    id_vars=["condition"],
    value_vars=["macro_f1", "fake_pr_auc", "fake_f1", "accuracy"],
    var_name="metric",
    value_name="value",
)
fig, axis = plt.subplots(figsize=(9, 5))
sns.barplot(metric_plot, x="metric", y="value", hue="condition", ax=axis)
axis.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(REPORT_DIR / "test_comparison.png", dpi=200)
plt.close(fig)

analysis_columns = [
    "review_id", "safe_review", "model_text", "label", "prediction",
    "char_length", "token_count", "codemix_ratio", "promo_keyword_hits",
    "exact_dup_cluster_size", "near_duplicate_degree",
    "near_duplicate_cluster_size", "has_promo", "has_emoji",
    "has_url", "has_price",
]
analysis_frame = expanded_predictions.select(*analysis_columns).toPandas()
analysis_frame["error_type"] = "correct"
analysis_frame.loc[
    (analysis_frame["label"] == 0) & (analysis_frame["prediction"] == 1),
    "error_type",
] = "false_negative"
analysis_frame.loc[
    (analysis_frame["label"] == 1) & (analysis_frame["prediction"] == 0),
    "error_type",
] = "false_positive"

error_summary = (
    analysis_frame.groupby("error_type")
    .agg(
        rows=("review_id", "size"),
        mean_length=("char_length", "mean"),
        mean_tokens=("token_count", "mean"),
        mean_codemix=("codemix_ratio", "mean"),
        mean_promotional_hits=("promo_keyword_hits", "mean"),
        mean_near_duplicate_degree=("near_duplicate_degree", "mean"),
    )
    .reset_index()
)
error_summary.to_csv(REPORT_DIR / "error_analysis.csv", index=False)

plot_features = [
    "char_length", "token_count", "codemix_ratio",
    "promo_keyword_hits", "near_duplicate_degree", "has_promo",
]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for axis, feature in zip(axes.flat, plot_features):
    sns.histplot(
        analysis_frame,
        x=feature,
        hue="error_type",
        element="step",
        common_norm=False,
        ax=axis,
    )
fig.tight_layout()
fig.savefig(REPORT_DIR / "error_analysis.png", dpi=200)
plt.close(fig)


In [ ]:
HANDCRAFTED_FEATURES = [
    "char_length", "token_count", "codemix_ratio", "repeated_char_ratio",
    "url_count", "email_count", "phone_count", "price_count",
    "hashtag_count", "mention_count", "exclamation_count", "emoji_count",
    "promo_keyword_hits", "bengali_chars", "latin_chars", "digit_chars",
    "other_chars", "exact_dup_cluster_size", "is_exact_duplicate",
    "near_duplicate_degree", "near_duplicate_similarity",
    "near_duplicate_cluster_size", "has_url", "has_email", "has_phone",
    "has_price", "has_promo", "has_emoji",
]

count_vectorizer_model = CountVectorizerModel.load(
    str(MODELS_DIR / "count_vectorizer")
)
vocabulary = count_vectorizer_model.vocabulary

logistic_gold = LogisticRegressionModel.load(
    str(MODELS_DIR / "logistic_regression_gold")
)
logistic_expanded = LogisticRegressionModel.load(
    str(MODELS_DIR / "logistic_regression_expanded")
)
linear_feature_names = vocabulary + HANDCRAFTED_FEATURES
if len(linear_feature_names) != logistic_expanded.coefficients.size:
    raise AssertionError("Logistic coefficient names do not match the feature vector")

coefficient_frame = pd.DataFrame({
    "feature": linear_feature_names,
    "gold_coefficient": logistic_gold.coefficients.toArray(),
    "expanded_coefficient": logistic_expanded.coefficients.toArray(),
})
coefficient_frame["absolute_expanded_coefficient"] = (
    coefficient_frame["expanded_coefficient"].abs()
)
coefficient_frame = coefficient_frame.sort_values(
    "absolute_expanded_coefficient",
    ascending=False,
)
coefficient_frame.to_csv(
    REPORT_DIR / "logistic_coefficients.csv",
    index=False,
)

tree_feature_names = [
    f"word2vec_{index}" for index in range(100)
] + HANDCRAFTED_FEATURES
random_forest = RandomForestClassificationModel.load(
    str(MODELS_DIR / "random_forest_expanded")
)
gbt = GBTClassificationModel.load(str(MODELS_DIR / "gbt_expanded"))

for name, model in (("random_forest", random_forest), ("gbt", gbt)):
    importance = model.featureImportances.toArray()
    if len(tree_feature_names) != len(importance):
        raise AssertionError(f"{name} feature names do not match its vector")
    pd.DataFrame({
        "feature": tree_feature_names,
        "importance": importance,
    }).sort_values("importance", ascending=False).to_csv(
        REPORT_DIR / f"{name}_feature_importance.csv",
        index=False,
    )


In [ ]:
TOKEN_PATTERN = re.compile(r"[\u0980-\u09ff]+|[A-Za-z]+|[০-৯\d]+", re.UNICODE)
BANGLA_STOPWORDS = {
    "আমি", "আমরা", "তুমি", "আপনি", "সে", "তিনি", "তারা", "এই", "এটা", "ওটা",
    "এবং", "আর", "বা", "কিন্তু", "যে", "কি", "না", "হয়", "হয়", "ছিল", "আছে",
    "থেকে", "জন্য", "সাথে", "একটি", "একটা", "এর", "তে", "কে", "ও", "তো",
}
ENGLISH_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from",
    "has", "have", "he", "i", "in", "is", "it", "of", "on", "or", "that",
    "the", "this", "to", "was", "we", "were", "with", "you",
}

_LIME_TOKENIZER = None

@F.udf(ArrayType(StringType()))
def lime_tokens(text):
    global _LIME_TOKENIZER
    try:
        if _LIME_TOKENIZER is None:
            from bnlp import BasicTokenizer
            _LIME_TOKENIZER = BasicTokenizer()
        raw_tokens = _LIME_TOKENIZER.tokenize(str(text or ""))
    except Exception:
        _LIME_TOKENIZER = False
        raw_tokens = TOKEN_PATTERN.findall(str(text or ""))
    stopwords = BANGLA_STOPWORDS | ENGLISH_STOPWORDS
    return [
        str(token).strip().lower()
        for token in raw_tokens
        if str(token).strip()
        and TOKEN_PATTERN.fullmatch(str(token).strip().lower())
        and str(token).strip().lower() not in stopwords
    ]

idf_model = IDFModel.load(str(MODELS_DIR / "idf"))
word2vec_model = Word2VecModel.load(str(MODELS_DIR / "word2vec"))

def prediction_function(model, model_name, context_vector):
    if model_name in ("logistic_regression", "linear_svc"):
        context_column = "handcrafted_scaled"
        feature_column = "features_linear"
    elif model_name == "naive_bayes":
        context_column = "handcrafted_nonnegative"
        feature_column = "features_nb"
    else:
        context_column = "handcrafted_raw"
        feature_column = "features_tree"

    schema = StructType([
        StructField("_index", IntegerType(), False),
        StructField("model_text", StringType(), False),
        StructField(context_column, VectorUDT(), False),
    ])
    context = Vectors.dense(context_vector.toArray())

    def predict(texts):
        rows = [
            (index, str(text), context)
            for index, text in enumerate(texts)
        ]
        frame = spark.createDataFrame(rows, schema=schema)
        frame = frame.withColumn("tokens", lime_tokens("model_text"))
        frame = count_vectorizer_model.transform(frame)

        if model_name in ("logistic_regression", "linear_svc"):
            frame = idf_model.transform(frame)
            frame = VectorAssembler(
                inputCols=["tfidf", context_column],
                outputCol=feature_column,
            ).transform(frame)
        elif model_name == "naive_bayes":
            frame = VectorAssembler(
                inputCols=["term_frequency", context_column],
                outputCol=feature_column,
            ).transform(frame)
        else:
            frame = word2vec_model.transform(frame)
            frame = VectorAssembler(
                inputCols=["word2vec", context_column],
                outputCol=feature_column,
            ).transform(frame)

        predictions = model.transform(frame).orderBy("_index")
        if "probability" in predictions.columns:
            return np.asarray([
                [float(row["probability"][0]), float(row["probability"][1])]
                for row in predictions.select("probability").collect()
            ])

        margins = [
            float(row["rawPrediction"][1])
            for row in predictions.select("rawPrediction").collect()
        ]
        probabilities = [
            1.0 / (1.0 + math.exp(-max(min(margin, 30.0), -30.0)))
            for margin in margins
        ]
        return np.asarray([[1.0 - value, value] for value in probabilities])

    return predict


In [ ]:
context_column = {
    "logistic_regression": "handcrafted_scaled",
    "linear_svc": "handcrafted_scaled",
    "naive_bayes": "handcrafted_nonnegative",
    "random_forest": "handcrafted_raw",
    "gbt": "handcrafted_raw",
}[selected_name]

false_negatives = (
    expanded_predictions
    .filter((F.col("label") == 0) & (F.col("prediction") == 1))
    .orderBy("review_id")
    .limit(5)
)
false_positives = (
    expanded_predictions
    .filter((F.col("label") == 1) & (F.col("prediction") == 0))
    .orderBy("review_id")
    .limit(5)
)
selected_errors = false_negatives.unionByName(false_positives)
remaining = max(15 - selected_errors.count(), 0)
selected_correct = (
    expanded_predictions
    .filter(F.col("label") == F.col("prediction"))
    .orderBy("review_id")
    .limit(remaining)
)
lime_cases = selected_errors.unionByName(selected_correct).select(
    "review_id", "model_text", "label", "prediction", context_column
).collect()

explainer = LimeTextExplainer(
    class_names=["Fake", "Genuine"],
    random_state=42,
)
lime_output = []
for row in lime_cases:
    predict = prediction_function(
        expanded_model,
        selected_name,
        row[context_column],
    )
    explanation = explainer.explain_instance(
        row["model_text"],
        predict,
        labels=(0, 1),
        num_features=10,
        num_samples=400,
    )
    predicted_label = int(row["prediction"])
    lime_output.append({
        "review_id": row["review_id"],
        "true_label": int(row["label"]),
        "predicted_label": predicted_label,
        "correct": int(row["label"]) == predicted_label,
        "influential_terms": explanation.as_list(label=predicted_label),
        "probability_note": (
            "sigmoid-transformed margin"
            if selected_name == "linear_svc"
            else "model probability"
        ),
    })

with open(REPORT_DIR / "lime_explanations.json", "w", encoding="utf-8") as handle:
    json.dump(lime_output, handle, ensure_ascii=False, indent=2)


In [ ]:
gold_result = test_metrics[0]
expanded_result = test_metrics[1]
test_class_counts = {
    int(row["label"]): int(row["count"])
    for row in test.groupBy("label").count().collect()
}
final_summary = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "selection": final_manifest,
    "test_rows": test.count(),
    "test_class_counts": test_class_counts,
    "test_scoring_policy": (
        "Validation-selected gold-only and expanded configurations were each "
        "scored once on the locked gold test split."
    ),
    "gold_only": gold_result,
    "gold_plus_pseudo": expanded_result,
    "delta_macro_f1": (
        expanded_result["macro_f1"] - gold_result["macro_f1"]
    ),
    "delta_fake_pr_auc": (
        expanded_result["fake_pr_auc"] - gold_result["fake_pr_auc"]
    ),
    "lime_cases": len(lime_output),
}
with open(REPORT_DIR / "final_summary.json", "w", encoding="utf-8") as handle:
    json.dump(final_summary, handle, ensure_ascii=False, indent=2)

with open(AUDIT_DIR / "final_evaluation_audit.json", "w", encoding="utf-8") as handle:
    json.dump({
        "created_at": final_summary["created_at"],
        "selection_source": "gold validation split",
        "locked_test_evaluations": ["gold_only", "gold_plus_pseudo"],
        "reported_metrics": [
            "macro_f1", "fake_pr_auc", "macro_precision", "macro_recall",
            "accuracy", "fake_roc_auc", "classwise_precision_recall_f1",
        ],
        "lime_cases": len(lime_output),
    }, handle, indent=2)

print(json.dumps(final_summary, ensure_ascii=False, indent=2))
